In [4]:
# Re-read dicom_3d.py on every cell run, so editing it doesn't need a
# kernel restart (a plain re-import won't: Python caches sys.modules).
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# Walk up to the repo root (the folder holding .git) so this works from any cwd.
ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "data"
sys.path.append(str(ROOT / "Aux_functions"))
sys.path.append(str(ROOT / "functions"))

# Force a re-read from disk. A kernel that already imported an older
# dicom_3d keeps it in sys.modules, and a plain `from ... import` won't
# notice the file changed -- that is what an ImportError here means.
import importlib
import dicom_utils3, dicom_3d
importlib.reload(dicom_utils3)
importlib.reload(dicom_3d)

from dicom_utils3 import iter_patient_studies, get_study_series_summary, load_series
from dicom_3d import select_largest_series, get_volume_with_spacing, plot_3d_volume, plot_sequences_separately, fuse_and_plot

import plotly.io as pio
# Only matters if you pass inline=True to plot_3d_volume. "notebook_connected"
# pulls plotly.js from the CDN instead of embedding ~3.5 MB of it per figure.
pio.renderers.default = "notebook_connected"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
# One study from the repo's own data/ dir: 3 sagittal + 1 axial + 1 coronal series.
STUDY = DATA / "img_sequences" / "1.2.826.0.1.3680043.8.498.10025765742726180988537955593315007559"

# Set this to a folder of .dcm slices -- here the sagittal PD FatSat (26 slices).
patient_folder = STUDY / "1.2.826.0.1.3680043.8.498.47103829651127852726285439804808053437"

In [6]:
datasets = load_series(patient_folder)
print(f"{len(datasets)} slices loaded")

volume, spacing = get_volume_with_spacing(datasets)
print("volume shape:", volume.shape, "spacing (dz, dy, dx):", spacing)

plot_3d_volume(volume, spacing, title=patient_folder.name)

26 slices loaded
volume shape: (26, 640, 640) spacing (dz, dy, dx): (3.900006929940694, 0.2813, 0.2813)
54,194 verts / 106,566 faces -> /Users/niko/Documents/DEV/2026-2028/SaveTheKnees/data/renders/1.2.826.0.1.3680043.8.498.47103829651127852726285439804808053437.html (4.7 MB)


PosixPath('/Users/niko/Documents/DEV/2026-2028/SaveTheKnees/data/renders/1.2.826.0.1.3680043.8.498.47103829651127852726285439804808053437.html')

In [7]:
series_folders = [
    STUDY / "1.2.826.0.1.3680043.8.498.12296387404690065979747592907647620416",  # SG T1W       (26 slices)
    STUDY / "1.2.826.0.1.3680043.8.498.17927783255029019223687232936481918350",  # SG T2W FS    (26 slices)
    STUDY / "1.2.826.0.1.3680043.8.498.47103829651127852726285439804808053437",  # SG PD FatSat (26 slices)
    STUDY / "1.2.826.0.1.3680043.8.498.52503045794545563216216974185473269584",  # AX T2W FS    (40 slices)
    STUDY / "1.2.826.0.1.3680043.8.498.58180411352017518670923060837962824092",  # CO T2W FS    (41 slices)
]

In [8]:
plot_sequences_separately(series_folders)


SG T1W: 26 slices, volume (26, 672, 672)
58,027 verts / 113,826 faces -> /Users/niko/Documents/DEV/2026-2028/SaveTheKnees/data/renders/SG_T1W.html (5.1 MB)

SG T2W FS: 26 slices, volume (26, 512, 512)
41,008 verts / 80,502 faces -> /Users/niko/Documents/DEV/2026-2028/SaveTheKnees/data/renders/SG_T2W_FS.html (3.6 MB)

SG PD FatSat: 26 slices, volume (26, 640, 640)
54,194 verts / 106,566 faces -> /Users/niko/Documents/DEV/2026-2028/SaveTheKnees/data/renders/SG_PD_FatSat.html (4.7 MB)

AX T2W FS: 40 slices, volume (40, 512, 512)
38,669 verts / 76,028 faces -> /Users/niko/Documents/DEV/2026-2028/SaveTheKnees/data/renders/AX_T2W_FS.html (3.4 MB)

CO T2W FS: 41 slices, volume (41, 512, 512)
34,297 verts / 67,724 faces -> /Users/niko/Documents/DEV/2026-2028/SaveTheKnees/data/renders/CO_T2W_FS.html (2.9 MB)


[PosixPath('/Users/niko/Documents/DEV/2026-2028/SaveTheKnees/data/renders/SG_T1W.html'),
 PosixPath('/Users/niko/Documents/DEV/2026-2028/SaveTheKnees/data/renders/SG_T2W_FS.html'),
 PosixPath('/Users/niko/Documents/DEV/2026-2028/SaveTheKnees/data/renders/SG_PD_FatSat.html'),
 PosixPath('/Users/niko/Documents/DEV/2026-2028/SaveTheKnees/data/renders/AX_T2W_FS.html'),
 PosixPath('/Users/niko/Documents/DEV/2026-2028/SaveTheKnees/data/renders/CO_T2W_FS.html')]

In [9]:
fuse_and_plot(
    series_folders,
    target_spacing=1.5,  # mm; coarser = faster, finer = more detail
    labels=["SG T1W", "SG T2W FS", "SG PD FatSat", "AX T2W FS", "CO T2W FS"],
)

SG T1W: (26, 672, 672)
SG T2W FS: (26, 512, 512)
SG PD FatSat: (26, 640, 640)
AX T2W FS: (40, 512, 512)
CO T2W FS: (41, 512, 512)

fusing onto (159, 163, 137) grid at 1.5 mm isotropic
coverage: 53% of the grid seen by >=1 series, 19% by all
40,432 verts / 80,964 faces -> /Users/niko/Documents/DEV/2026-2028/SaveTheKnees/data/renders/fused__SG_T1W___SG_T2W_FS___SG_PD_FatSat___AX_T2W_FS___CO_T2W_FS.html (2.6 MB)


PosixPath('/Users/niko/Documents/DEV/2026-2028/SaveTheKnees/data/renders/fused__SG_T1W___SG_T2W_FS___SG_PD_FatSat___AX_T2W_FS___CO_T2W_FS.html')